In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U ultralytics

In [ ]:
MODEL_NAME = "yolo11l"      # yolo11l 
WEIGHTS    = "yolo11l.pt"   # yolo11l.pt
IS_RTDETR  = False          # True only in the RT-DETR notebook

EPOCHS   = 100
PATIENCE = 30
IMGSZ    = 640
BATCH    = 16              
SEED     = 0

In [ ]:
from pathlib import Path

NAMES = ['trash_bag', 'trash_bottle', 'trash_branch', 'trash_can', 'trash_clothing',
         'trash_container', 'trash_cup', 'trash_net', 'trash_pipe', 'trash_rope',
         'trash_snack_wrapper', 'trash_tarp', 'trash_unknown_instance', 'trash_wreckage']

def find_dir(name):
    for p in Path("/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data").rglob(name):
        if p.is_dir() and (p / "images").exists():
            return p
    raise FileNotFoundError(name)

def write_yaml(path, root, train, val, test):
    with open(path, "w") as f:
        f.write(f"path: {root}\ntrain: {train}\nval: {val}\ntest: {test}\nnames:\n")
        for i, n in enumerate(NAMES):
            f.write(f"  {i}: {n}\n")

Y = Path("/kaggle/working/yaml"); Y.mkdir(exist_ok=True)
yaml_main = Y / "main.yaml"
yaml_dim  = Y / "dim.yaml"
yaml_dark = Y / "dark.yaml"

write_yaml(yaml_main, find_dir("trashcan"), "images/train", "images/val", "images/test")
write_yaml(yaml_dim,  find_dir("trashcan_dim"),  "images/test", "images/test", "images/test")
write_yaml(yaml_dark, find_dir("trashcan_dark"), "images/test", "images/test", "images/test")
print(open(yaml_main).read())

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO, RTDETR

model = RTDETR(WEIGHTS) if IS_RTDETR else YOLO(WEIGHTS)
model.train(
    data=str(yaml_main), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
    batch=BATCH, seed=SEED, workers=4, device=0,
    project="/kaggle/working/runs", name=MODEL_NAME, exist_ok=True, plots=True,
)
save_dir = Path(model.trainer.save_dir)
print("Saved to:", save_dir)

In [ ]:
import pandas as pd, yaml, torch, ultralytics

OUT = Path(f"/kaggle/working/results/{MODEL_NAME}"); OUT.mkdir(parents=True, exist_ok=True)

# ---- 1) Hyperparameters ----
args = yaml.safe_load(open(save_dir / "args.yaml"))
extra = {
    "model_name": MODEL_NAME, "pretrained_weights": WEIGHTS,
    "num_parameters": sum(p.numel() for p in model.model.parameters()),
    "gpu": torch.cuda.get_device_name(0),
    "ultralytics_version": ultralytics.__version__, "torch_version": torch.__version__,
}
hp = pd.DataFrame(list({**extra, **args}.items()), columns=["parameter", "value"])
hp.to_csv(OUT / "hyperparameters.csv", index=False)

# ---- 2) Per-epoch metrics ----
df = pd.read_csv(save_dir / "results.csv")
df.columns = df.columns.str.strip()
P, R = "metrics/precision(B)", "metrics/recall(B)"
M50, M5095 = "metrics/mAP50(B)", "metrics/mAP50-95(B)"

df["f1_score"] = 2 * df[P] * df[R] / (df[P] + df[R] + 1e-9)
train_loss_cols = [c for c in df.columns if c.startswith("train/") and "loss" in c]
val_loss_cols   = [c for c in df.columns if c.startswith("val/") and "loss" in c]
df["train_total_loss"] = df[train_loss_cols].sum(axis=1)
df["val_total_loss"]   = df[val_loss_cols].sum(axis=1)
df["fitness"] = 0.1 * df[M50] + 0.9 * df[M5095]     # Ultralytics' rule for choosing best.pt
df.to_csv(OUT / "epoch_metrics.csv", index=False)

# ---- 3) Best epoch (loss), best epoch (fitness), final epoch ----
def pick(label, r):
    return {"which": label, "epoch": int(r["epoch"]), "precision": r[P], "recall": r[R],
            "f1_score": r["f1_score"], "mAP50": r[M50], "mAP50_95": r[M5095],
            "train_total_loss": r["train_total_loss"], "val_total_loss": r["val_total_loss"]}

summary = pd.DataFrame([
    pick("best_epoch_by_val_loss", df.loc[df["val_total_loss"].idxmin()]),
    pick("best_epoch_by_fitness (best.pt)", df.loc[df["fitness"].idxmax()]),
    pick("final_epoch", df.iloc[-1]),
])
summary.to_csv(OUT / "summary_epochs.csv", index=False)

# ---- 4) Test-set results: bright / dim / dark, using best.pt ----
best = save_dir / "weights" / "best.pt"
m = RTDETR(str(best)) if IS_RTDETR else YOLO(str(best))
test_rows, class_rows = [], []
for cond, y in [("bright", yaml_main), ("dim", yaml_dim), ("dark", yaml_dark)]:
    r = m.val(data=str(y), split="test", imgsz=IMGSZ, batch=BATCH, plots=False,
              project="/kaggle/working/runs", name=f"{MODEL_NAME}_test_{cond}", exist_ok=True)
    p, rc = float(r.box.mp), float(r.box.mr)
    test_rows.append({"condition": cond, "precision": p, "recall": rc,
                      "f1_score": 2 * p * rc / (p + rc + 1e-9),
                      "mAP50": float(r.box.map50), "mAP50_95": float(r.box.map)})
    for i, ap in enumerate(r.box.maps):
        class_rows.append({"condition": cond, "class": NAMES[i], "mAP50_95": float(ap)})

pd.DataFrame(test_rows).to_csv(OUT / "test_results.csv", index=False)
pd.DataFrame(class_rows).to_csv(OUT / "test_per_class.csv", index=False)

print(summary.round(4).to_string(index=False))
print(pd.DataFrame(test_rows).round(4).to_string(index=False))